# 📦 Step 1c — Download DeepSeek-V4-Pro-Qwen3.5-9B (uncensored, F16) to Kaggle Dataset

Downloads the **DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic** GGUF model from HuggingFace to a permanent private Kaggle Dataset, so the server can mount it instantly every session.

> This model is a Qwen3.5-9B reasoning model distilled from DeepSeek-V4-Pro, decensored with Heretic ARA. Architecture `qwen3_5` — requires llama.cpp **b12182+** (re-run `1b_setup_llama.ipynb` if unsure).

### Quantization choice (edit QUANT in Cell 2)
| Quant | Size | Notes |
|---|---|---|
| **F16** (default) | ~17.1 GB | **Full precision** — set as default per request. Fits in 30 GB VRAM when loaded alone |
| f16_q8_0 | ~13.9 GB | Near-F16, smaller |
| Q4_K_M | ~5.7 GB | Smallest/fastest, lowest quality |

**Exact filename note:** the real F16 file on HuggingFace is `DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-f16.gguf` (hyphen + lowercase `f16`) in the `Mungert` repo. A spelling like `...heretic.F16.gguf` (dot + uppercase) does **not** exist and will 404 — this notebook uses the correct name.

The F16 main-model repo has **no bundled mmproj**. The vision projector (`mmproj-...-f16.gguf`, ~0.9 GB) is downloaded from the companion `uberkie` repo and is only needed for image input. Set `DOWNLOAD_MMPROJ = False` in Cell 2 to skip it (text-only use).

## ⚙️ Cell 1 — Install download dependencies

In [ ]:
%%capture
!pip install -q huggingface_hub hf_transfer

## ⚙️ Cell 2 — Configuration

In [ ]:
import os, json
from pathlib import Path

# ── Repos ──────────────────────────────────────────────────────────────────────
# F16 main model lives in Mungert's repo; quants + mmproj live in uberkie's.
REPO_F16  = "Mungert/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-GGUF"
REPO_QUANT = "uberkie/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-GGUF"

# ── Pick quantization: F16 / f16_q8_0 / Q4_K_M ────────────────────────────────
QUANT = "F16"   # ← default: full precision 17.1 GB

# mmproj (vision projector) — from uberkie repo; skip if text-only
DOWNLOAD_MMPROJ = True

MODEL_BASE = "DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"
if QUANT == "F16":
    HF_REPO    = REPO_F16
    MODEL_FILE = f"{MODEL_BASE}-f16.gguf"          # exact HF name, 17.1 GB
elif QUANT == "f16_q8_0":
    HF_REPO    = REPO_F16
    MODEL_FILE = f"{MODEL_BASE}-f16_q8_0.gguf"     # 13.9 GB
else:
    HF_REPO    = REPO_QUANT
    MODEL_FILE = f"{MODEL_BASE}-{QUANT}.gguf"      # e.g. Q4_K_M 5.7 GB

MMPROJ_REPO = REPO_QUANT
MMPROJ_FILE = "mmproj-DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic-f16.gguf" if DOWNLOAD_MMPROJ else None

DATASET_SLUG = "deepseek-v4-pro-9b-gguf"   # your Kaggle dataset name
DOWNLOAD_DIR = Path("/tmp/deepseek9b")

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

print(f"🤗 Repo      : {HF_REPO}")
print(f"📦 Model     : {MODEL_FILE}")
print(f"🗜️  Quant     : {QUANT}")
print(f"👁  mmproj    : {MMPROJ_FILE or '(skipped)'}")
print(f"📁 Directory : {DOWNLOAD_DIR}")

## 📥 Cell 3 — Download from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download
import shutil

print("🚀 Downloading (F16 is ~17 GB, this can take 10–30 minutes)...\n")

def grab(filename, repo):
    cached = hf_hub_download(repo_id=repo, filename=filename, repo_type="model")
    dst = DOWNLOAD_DIR / filename
    if Path(cached).resolve() != dst.resolve():
        shutil.copy2(cached, dst)
    size_gb = dst.stat().st_size / (1024**3)
    print(f"  ✅ {filename}  ({size_gb:.2f} GB)")
    return dst

grab(MODEL_FILE, HF_REPO)
if MMPROJ_FILE:
    grab(MMPROJ_FILE, MMPROJ_REPO)
print("\n✅ Download complete.")

## 🔍 Cell 4 — Verify files

In [ ]:
for f in sorted(DOWNLOAD_DIR.glob("*.gguf")):
    print(f"  {f.name:<70} {f.stat().st_size/(1024**3):6.2f} GB")
assert (DOWNLOAD_DIR / MODEL_FILE).exists(), "Model file missing!"
print("\n✅ Files verified.")

## 📦 Cell 5 — Create private Kaggle Dataset

Username is auto-detected (env var → `~/.kaggle/kaggle.json`).

In [ ]:
import subprocess

KAGGLE_USERNAME = ""
def _detect_kaggle_username():
    u = os.environ.get("KAGGLE_USERNAME", "").strip()
    if u:
        return u
    u = os.environ.get("KAGGLE_USER", "").strip()
    if u:
        return u
    for token in (Path.home()/".kaggle"/"kaggle.json", Path("/root/.kaggle/kaggle.json")):
        try:
            if token.exists():
                d = json.loads(token.read_text())
                if d.get("username"):
                    return d["username"]
        except Exception:
            pass
    return ""

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = _detect_kaggle_username()
if not KAGGLE_USERNAME:
    raise ValueError("Set KAGGLE_USERNAME at top: KAGGLE_USERNAME = 'your_username'")

metadata = {
    "title"   : "DeepSeek-V4-Pro-Qwen3.5-9B GGUF",
    "id"      : f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "licenses": [{"name": "apache-2.0"}],
    "isPrivate": True,
}
(DOWNLOAD_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False))

print(f"👤 {KAGGLE_USERNAME}   📦 {KAGGLE_USERNAME}/{DATASET_SLUG}\n")
result = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(DOWNLOAD_DIR), "--dir-mode", "skip"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("⚠️ stderr:", result.stderr)
else:
    print(f"\n✅ Dataset created: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG}")

## ✅ Done!

In `2_run_server.ipynb`, add this dataset as an input. The server auto-detects the model files — no path editing needed.